In [ ]:
import pandas as pd
import os

# Step 1 - see what files are in the dataset
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/hanadithaisir/daily-market-prices/9ef84268-d588-465a-a308-a864a43d0070.csv')
print("shapes:",df.shape)
print("\ncolumns:",df.columns.tolist())
print("\nfirst 5 rows",df.head())
#print(df.head())

In [ ]:
df.columns=df.columns.str.replace('_x0020_',' ').str.strip()
df.columns = ['State', 'District', 'Market', 'Commodity', 
               'Variety', 'Grade', 'Arrival_Date', 
               'Min_Price', 'Max_Price', 'Modal_Price']
df['Arrival_Date']=pd.to_datetime(df['Arrival_Date'],dayfirst=True)
print("Missing values in each column:")
print(df.isnull().sum())

print("\nCleaned columns:", df.columns.tolist())
print("\nDate type now:", df['Arrival_Date'].dtype)


In [ ]:
print("States:",df['State'].nunique())
print("Markets:",df['Market'].nunique())
print("Commodities:",df['Commodity'].nunique())

print("\nAll commodities:")
print(sorted(df['Commodity'].unique()))

# Price summary
print("\nPrice summary:")
print(df[['Min_Price', 'Max_Price', 'Modal_Price']].describe())

In [ ]:
# Filter only Arecanut rows
arecanut = df[df['Commodity'] == 'Arecanut(Betelnut/Supari)']

print("Arecanut records:", len(arecanut))
print("\nArecanut by State:")
print(arecanut[['State', 'Market', 'Min_Price', 
                 'Max_Price', 'Modal_Price']])

In [ ]:
# Sort by Modal Price to see who pays most and least
print("Arecanut prices ranked - highest to lowest:")
print(arecanut[['State', 'Market', 'Modal_Price']]
      .sort_values('Modal_Price', ascending=False)
      .to_string(index=False))

# Price gap
highest = arecanut['Modal_Price'].max()
lowest = arecanut['Modal_Price'].min()
gap = highest - lowest

print(f"\nHighest price: ₹{highest:,.0f}")
print(f"Lowest price:  ₹{lowest:,.0f}")
print(f"Price gap:     ₹{gap:,.0f}")
print(f"A farmer in the lowest market loses ₹{gap:,.0f} per quintal")

In [ ]:
# Why does Shimoga appear 3 times with different prices?
shimoga = df[
    (df['Commodity'] == 'Arecanut(Betelnut/Supari)') & 
    (df['Market'].str.contains('Shimoga'))
]

print("Shimoga arecanut breakdown:")
print(shimoga[['Market', 'Variety', 'Grade', 
               'Min_Price', 'Max_Price', 'Modal_Price']]
      .to_string(index=False))

In [ ]:
# Top 10 most expensive commodities by average modal price
top_commodities = (df.groupby('Commodity')['Modal_Price']
                     .mean()
                     .sort_values(ascending=False)
                     .head(10)
                     .reset_index())

top_commodities.columns = ['Commodity', 'Avg_Modal_Price']
top_commodities['Avg_Modal_Price'] = top_commodities['Avg_Modal_Price'].round(0)

print("Top 10 most expensive commodities:")
print(top_commodities.to_string(index=False))

In [ ]:
# Save cleaned dataframe to a new CSV
df.to_csv('/kaggle/working/market_prices_cleaned.csv', index=False)
print("Saved successfully!")

# Also save arecanut specific data
arecanut.to_csv('/kaggle/working/arecanut_prices.csv', index=False)
print("Arecanut data saved!")